In [1]:
import torch
print(torch.__version__)

2.9.0+cpu


In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
#%load_ext autoreload
#%autoreload 2

from src import (
    GridSpec, HelmholtzConfig, SweepConfig,
    FiniteDifference,
    gmres_solve, GMRESOptions,
)
from src.visualisation import plot_residuals
from src.datasets import build_direct_map, build_freq_transfer
from src.models import SimpleFNO, LocalCNN
from src.training import train_model, show_example
from src.solvers import direct_solve

# PML & operator entry points come from src.operators
from src.operators import (
    PMLConfig,
    helmholtz_operator,
    laplacian_operator,
)


In [9]:
# Core API
import numpy as np
from functools import partial


from src.solvers import direct_solve  # your existing numerical oracle


In [ ]:
grid = GridSpec(dims=2, shape=(60, 60), lengths=(1.0, 1.0))
disc = FiniteDifference(FDConfig(bc=BC.DIRICHLET, dtype="complex128"))
gmres_opts = GMRESOptions(tol=1e-6, restart=None, maxiter=None)

K_LOW_RANGE  = (10.0, 30.0)
K_HIGH_RANGE = (40.0, 90.0)
N_SAMPLES    = 100  # 200–500 later


In [11]:
# after the reload you did:
#data_A = build_direct_map(
#    n=N_SAMPLES, grid=grid, disc=disc, k_range=K_LOW_RANGE,
#    solver_fn=s.direct_solve,   # <— use the reloaded function
#)

#data_B = build_freq_transfer(
#    n=N_SAMPLES, grid=grid, disc=disc,
#    k_low_range=K_LOW_RANGE, k_high_range=K_HIGH_RANGE,
#    solver_fn=s.direct_solve,   # <— same here
#)

In [12]:
import numpy as np
from src import GridSpec, GMRESOptions
from src.operators import PMLConfig

# Grid
grid = GridSpec(dims=2, shape=(60, 60), lengths=(1.0, 1.0))

# PML config (safe defaults)
nx, ny = grid.shape
hx, hy = grid.lengths[0]/(nx-1), grid.lengths[1]/(ny-1)
K_LOW_RANGE  = (10.0, 30.0)
K_HIGH_RANGE = (40.0, 90.0)
k_high_max   = float(max(K_HIGH_RANGE))
t_cells      = max(10, int(0.12 * min(nx, ny)))       # ~12% of side
sigma_max    = 0.8 * k_high_max / min(hx, hy)         # scale with k/h
pml_cfg      = PMLConfig(thickness=t_cells, m=3, sigma_max=sigma_max)

# GMRES options (if/where you still use GMRES)
gmres_opts = GMRESOptions(tol=1e-6, restart=None, maxiter=None)

N_SAMPLES = 100  # 200–500 later


In [ ]:
#mini_A = build_direct_map(n=2, grid=grid, disc=disc, k_range=K_LOW_RANGE, solver_fn=s.direct_solve)
#print(data_A["x"].shape, data_A["y"].shape)
#print(data_B["x"].shape, data_B["y"].shape)


In [15]:
# --- tiny sanity dataset using the PML-backed solver ---

from functools import partial
import numpy as np
from src.solvers import solve_helmholtz_field  # returns np.ndarray
# assumes you've already defined: grid, K_LOW_RANGE, and pml_cfg (PMLConfig)

# Bind grid + PML once; switch to pml=None for Dirichlet runs
solver_truth = partial(
    solve_helmholtz_field,
    shape=grid.shape,
    lengths=grid.lengths,
    dtype=np.complex128,
    pml=pml_cfg,      # <- set to None for Dirichlet
)

mini_A = build_direct_map(
    n=2,
    grid=grid,
    disc=None,            # disc not used when solver_fn assembles internally
    k_range=K_LOW_RANGE,
    solver_fn=solver_truth,
)

print("mini_A:", mini_A["x"].shape, mini_A["y"].shape)

# If you already built data_A / data_B earlier, print their shapes too
try:
    print("data_A:", data_A["x"].shape, data_A["y"].shape)
    print("data_B:", data_B["x"].shape, data_B["y"].shape)
except NameError:
    pass


AttributeError: 'NoneType' object has no attribute 'assemble'

In [ ]:
from functools import partial
from src.solvers import direct_solve_pml

solver_truth = partial(
    direct_solve_pml,
    shape=grid.shape,
    lengths=grid.lengths,
    pml=pml_cfg,
)


In [ ]:
show_example(model_A, data_A, index=0, title="(A) Direct map (f,k)→u")
show_example(model_B, data_B, index=0, title="(B) Frequency transfer (u_low,k_high)→u_high")
